# 3 · Representations, targets and splits

Stage three of five. Takes the raw h5ad that [stage 1](1_data.ipynb) wrote and turns it into the
file every model reads: `..._with_targets_<score>.h5ad`, carrying both cell representations, the
response matrix, and the train/val/test assignment.

Four steps, in this order and for a reason:

| | step | writes | why it must come after the previous |
|---|---|---|---|
| **A** | `scgpt` | `obsm['X_scGPT']` | needs the raw h5ad |
| **B** | `targets` | `obsm['Y_ctrp']`, `obsm['M_ctrp']`, `uns['ctrp_drugs']` | writes into the embedding file |
| **C** | `splits` | `obs['split_ctrp']` | eligibility is derived from `M_ctrp` |
| **D** | `pca` | `obsm['X_pca']`, `obsm['X_pca_train_ctrp']` | the train-only fit needs `split_ctrp` |

**[Stage 2](2_drug_selection.ipynb) has already run and is not consumed here.** The drug panel is
applied at *training* time, in [stage 4](4a_percell_training.ipynb) — see §B for why that is a decision and
not an accident.

> ⛔ **Nothing here has been re-run.** The 03.08.2026 freeze in [TODO](../docs/TODO.md) holds until
> Selin's review finishes. Every artifact on disk predates the code that now writes it — the
> embeddings, `X_pca`, and every run under `runs/`. This notebook defines the stage; it is not a
> record of a run.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import DEFAULT_CTRP_SCORE, DEFAULT_VARIANT, PipelinePaths
from scripts.preprocessing import pipeline

VARIANT = DEFAULT_VARIANT
SCORE = DEFAULT_CTRP_SCORE          # auc_cc; ln_ic50_cc writes its own targets h5ad

# Machine-specific, and deliberately not in layout.py: scGPT lives in a separate checkout and
# virtualenv on whichever machine runs this, so it is not a property of the project layout.
# Set it to the interpreter that has `scgpt` installed.
SCGPT_PYTHON = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'

paths = PipelinePaths.build(None, VARIANT, SCORE)
print(f'variant : {paths.variant} -> {paths.processed_dir}')
print(f'score   : {paths.score} -> {paths.targets_h5ad.name}')
print(f'input   : {paths.raw_h5ad.name}  ({"present" if paths.raw_h5ad.exists() else "MISSING - run stage 1"})')

## A · scGPT embedding

scGPT pins versions this project does not, so it lives in its own virtualenv and this step is a
**subprocess**, not an import. `SCGPT_PYTHON` above is that interpreter.

The only operation applied to expression before the model reads it is a binning of each cell into 51
bins whose edges are quantiles of **that same cell's** non-zero values. Nothing is normalised or
log-transformed on this path, and nothing needs to be: binning is a rank transform, so dividing by
library size or taking a logarithm moves the values and the bin edges together and every gene lands
in the bin it started in. Normalising before scGPT is not merely unnecessary — it is inert.

Genes outside scGPT's vocabulary are dropped from `.X` in the file this writes.
⛔ That drop is **not** clean: most of it was a symbol-matching defect rather than genuine vocabulary
coverage, repaired in the pipeline but **not yet reflected in any artifact on disk**
([Corrections](../docs/steps/corrections-and-dead-ends.md#scgpt-discarded-genes-that-are-in-its-vocabulary-under-their-current-symbols)).

⚠️ **No interactive fallback.** Without `SCGPT_PYTHON` this raises and prints the command to run by
hand — it does not prompt. A blocking prompt is invisible under `jupyter nbconvert --execute` and
would hang rather than fail. If you do run it by hand, continue at §B: re-running §A afterwards
raises, because the embedding it would write now exists.

In [ ]:
embed_h5ad = pipeline.scgpt(paths, SCGPT_PYTHON)
embed_h5ad

## B · Response targets

Writes `Y_ctrp` (cell × drug), its observation mask `M_ctrp`, and the column order in
`uns['ctrp_drugs']`, for every drug CTRPv2 screened against at least `min_cell_lines` of the cell
lines that overlap SCP542.

**The drug panel does not enter here — decided 12.08.2026 (Selin).** [Stage 2](2_drug_selection.ipynb)
runs before this one and its `panel.csv` *could* be passed to `ctrp_to_h5ad --drugs`. It is not,
because the panel's whole effect is on the **model**: it sets `output_dim`, that is, how many heads
share one trunk. It says what the network is asked to predict, not what the data contains. So
`Y_ctrp` / `M_ctrp` keep the full screened catalog and the panel is applied as a column selection in
[stage 4](4a_percell_training.ipynb), via `MultiDrugDataset(drugs=…)`. Written up in
[Step 03](../docs/steps/03-model-and-training-design.md#the-drug-panel-is-a-training-time-choice-not-a-property-of-the-target-file-12082026).

Two consequences of that, neither of them the reason for it:

- **Changing the panel needs no preprocessing re-run.** One targets h5ad serves any panel, which is
  why stage 2 can rebuild the panel under the freeze while these h5ads cannot be rebuilt.
- **Filtering here would have moved the splits.** §C decides eligibility from *"this line has at
  least one observed label"* — a test taken over the *width* of `M`. Narrowing `M` from ~545 columns
  to 11 would re-evaluate it against the panel and could drop lines that are screened but not against
  a panel compound, silently redrawing `split_ctrp`.

The measure is whichever `SCORE` names. `auc_cc` is complete for every curve; `ln_ic50_cc` is
missing for ~40 % of them by construction, because an IC50 that falls well outside the measured dose
range is discarded — which is most compounds that never reach half-killing.

In [ ]:
targets_h5ad = pipeline.targets(paths)
targets_h5ad

## C · Train / validation / test split

A 70/15/15 partition **of cell lines**, so every cell of a line falls on the same side. The label is
defined per (cell line × drug) and broadcast to every cell of that line, so a per-cell split would
put copies of the same label on both sides — the leak that forced this design
([Step 04](../docs/steps/04-single-task-results.md)).

**The assignment is frozen to `splits/split_ctrp.csv`, not redrawn.** Eligibility depends on which
drugs survive the filters, so redrawing would move lines between train, validation and test whenever
anything upstream changed, and results from either side of that change would look comparable while
being scored on different held-out lines. A line the file does not cover is an error, not a fresh
draw.

⚠️ `regenerate=True` redraws and overwrites it. **Every result produced before it was regenerated was
scored on different held-out lines.** That is not a repair.

In [ ]:
pipeline.splits(paths)

## D · PCA baseline

Fitted on the **convert output** — the full HVG set from [stage 1](1_data.ipynb) — and *not* on this
file's own `.X`, from which §A has dropped out-of-vocabulary genes. Both representations therefore
rest on the same single gene filter, which is what makes the comparison one of representations
rather than of gene sets.

Two keys are written, and the difference matters:

- **`X_pca`** — fitted on every cell. Descriptive: correct for UMAPs, **wrong as model input**,
  because a held-out cell's coordinates then depend on held-out cells.
- **`X_pca_train_ctrp`** — fitted on the training lines of the frozen split alone, per-gene
  standardisation and rotation both, then applied to the held-out lines. This is what a model scored
  on that split reads.

scGPT needs no such distinction: its weights are pretrained and frozen and its binning uses only the
cell being embedded, so nothing about it is fitted on this data.

Runs last because the train-only fit needs the `split_ctrp` column §C writes.

In [ ]:
pipeline.pca(paths)

In [ ]:
import anndata as ad
import numpy as np

a = ad.read_h5ad(targets_h5ad, backed='r')
groups = a.obs['Cell_line'].astype(str).to_numpy()
split = a.obs['split_ctrp'].astype(str).to_numpy()
first = {ln: split[groups == ln][0] for ln in np.unique(groups)}

print(f'cells        : {a.n_obs:,}')
print(f'obsm         : {sorted(a.obsm.keys())}')
print(f'K (drugs)    : {len(a.uns["ctrp_drugs"])}   score={a.uns["ctrp_score"]}')
print(f'observed     : {int(np.asarray(a.obsm["M_ctrp"]).sum()):,} (cell x drug) labels')
for s in ('train', 'val', 'test', 'unassigned'):
    print(f'  {s:11s}: {sum(v == s for v in first.values()):3d} cell lines')
a.file.close()